# GenerativeAURORA — Step 10: Ablation Study

### Purpose
Step 10 systematically tests the contribution of the model components and produces **Table VI** for the paper.

### Workflow position
**Step 5 — Train → Step 6 — Evaluate → Step 7 — RCCC → Step 8 — Figures → Step 9 — Main paper tables → Step 10 — Ablation study**

### Ablation variants in the original script
- **A1:** Full model + RCCC
- **A2:** Full model, no RCCC
- **A3:** No regime conditioning
- **A4:** No condition encoder
- **A5:** No site embedding
- **A6:** Gaussian baseline

### Faithful-conversion rule
The supplied `step10_ablation.py` is the source of truth. The Python code in this notebook is copied **verbatim** and only divided into logical notebook cells. No model component, ablation definition, metric, parameter, result, or output path is silently changed.

The Markdown cells below explain what each original section is doing without altering the methodology.


## 1. Header and imports

This opening block is preserved exactly from the source script. It states the purpose of Step 10, the six ablation variants, the reported metrics, and the original command-line execution instruction.

In [ ]:
# GenerativeAURORA - Step 10: Ablation Study
#
# Tests the contribution of each model component systematically.
# This produces Table VI in the paper (ablation table).
#
# Ablation variants tested:
#   A1. Full model + RCCC          (GenerativeAURORA + RCCC) -- proposed
#   A2. Full model, no RCCC        (GenerativeAURORA)        -- ablate RCCC
#   A3. No regime conditioning     (DDPM no regime)          -- ablate sky gating
#   A4. No condition encoder       (blind DDPM)              -- ablate weather context
#   A5. No site embedding          (site-agnostic DDPM)      -- ablate site identity
#   A6. Gaussian baseline          (trivial)                 -- lower bound
#
# For each variant we report:
#   CRPS, Coverage 90% (overall), Coverage 90% Overcast, Median MAE
#
# Run:
#   python step10_ablation.py

import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import math
from torch.utils.data import DataLoader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Dataset and evaluation paths

Defines the dataset, evaluation, and paper-table directories. The original Windows paths are intentionally preserved in this faithful conversion.

In [ ]:
DATASET_DIR = "/content/drive/MyDrive/Colab Notebooks/Dataset"
CODING_DIR  = "/content/drive/MyDrive/Colab Notebooks/GenerativeAURORA_step4_build_model_Colab.ipynb"
EVAL_DIR    = os.path.join(DATASET_DIR, "evaluation")
TABLE_DIR   = os.path.join(DATASET_DIR, "paper_tables")
os.makedirs(TABLE_DIR, exist_ok=True)

## 3. Ablation parameters

Preserves the original scenario count, batch size, horizon, lookback, feature count, regime/site counts, embedding dimensions, diffusion steps, and RCCC alpha.

In [ ]:
N_SCENARIOS = 50
BATCH_SIZE  = 32
HORIZON     = 16
LOOKBACK    = 16
N_FEATURES  = 10
N_REGIMES   = 3
N_SITES     = 5
EMBED_DIM   = 64
HIDDEN_DIM  = 128
T_DIFFUSION = 200
ALPHA       = 0.10

## 4. Regime labels

Defines the original sky-regime ID-to-name mapping used by the ablation analysis.

In [ ]:
REGIME_NAMES = {0: "Clear", 1: "Cloudy", 2: "Overcast"}

import sys
sys.path.insert(0, CODING_DIR)

## 5. Step 4 model import

Imports the complete GenerativeAURORA model components directly from `step4_build_model.py`, exactly as required by the source script.

In [ ]:
# -----------------------------------------------------------------------
# IMPORT MODEL FROM STEP 4
# -----------------------------------------------------------------------
import os
import json
import sys

# Path to the target notebook on Google Drive
notebook_path = "/content/drive/MyDrive/Colab Notebooks/GenerativeAURORA_step4_build_model_Colab.ipynb"

if not os.path.exists(notebook_path):
    raise FileNotFoundError(f"Could not find notebook at: {notebook_path}. Please check your Google Drive mount or path.")

# Read and execute Python code blocks from the step 4 notebook directly
with open(notebook_path, 'r', encoding='utf-8') as f:
    nb_data = json.load(f)

# Execute code cells sequentially in the global namespace
for cell in nb_data.get('cells', []):
    if cell.get('cell_type') == 'code':
        code_text = ''.join(cell.get('source', []))
        # Avoid executing installations or imports of itself recursively
        if "import_ipynb" not in code_text and "GenerativeAURORA_step4_build_model_Colab" not in code_text:
            exec(code_text, globals())

PyTorch version: 2.11.0+cu130
CUDA available  : True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HORIZON     : 16
LOOKBACK    : 16
N_FEATURES  : 10
N_REGIMES   : 3
N_SITES     : 5
T_DIFFUSION : 200
EMBED_DIM   : 64
HIDDEN_DIM  : 128
Betas shape     : (200,)
Alphas shape    : (200,)
Alpha-bar shape : (200,)
First beta      : 9.999999747378752e-05
Last beta       : 0.019999999552965164
Device              : cuda
Total parameters    : 5,095,329
Trainable parameters: 5,095,329
x0 shape       : (4, 16)
cond_x shape   : (4, 16, 10)
regime shape   : (4,)
site shape     : (4,)
Loss value     : 0.981605
Loss dimensions: () (scalar expected)

Forward-pass shape check: PASSED
Generated scenarios shape: (4, 16)
Expected shape           : (4, 16)
Scenario-generation shape check: PASSED


## 6. Dataset class

Loads each sequence sample with its input sequence, target GHI, regime label, and site label, matching the earlier dataset stages.

In [ ]:
class SolarSequenceDataset(torch.utils.data.Dataset):
    def __init__(self, npz_path):
        data        = np.load(npz_path)
        self.X      = torch.tensor(data["X"],      dtype=torch.float32)
        self.Y      = torch.tensor(data["Y"],      dtype=torch.float32)
        self.regime = torch.tensor(data["regime"], dtype=torch.long)
        self.site   = torch.tensor(data["site"],   dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.regime[idx], self.site[idx]

## 7. Scaling utilities

Loads the previously fitted GHI scaler and converts normalized GHI back to physical W/m² units.

In [ ]:
def load_scalers():
    with open(os.path.join(DATASET_DIR, "scaler_params.json"), "r", encoding="utf-8") as f:
        return json.load(f)


def inv_scale(arr, scalers):
    s = scalers["GHI"]
    return arr * (s["max"] - s["min"]) + s["min"]

## 8. No-condition-encoder ablation

Implements the A4 ablation by replacing the condition encoder output with zeros, so the model receives no weather-context representation.

In [ ]:
# -----------------------------------------------------------------------
# ABLATION VARIANT: NO CONDITION ENCODER
# Replace ConditionEncoder output with zeros -- model gets no weather context
# -----------------------------------------------------------------------
class BlindConditionEncoder(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.embed_dim = embed_dim

    def forward(self, x):
        B = x.shape[0]
        return torch.zeros(B, self.embed_dim, device=x.device)


class GenerativeAURORA_NoContext(GenerativeAURORA):
    def __init__(self, T=T_DIFFUSION):
        super().__init__(T=T)
        self.unet.cond_enc = BlindConditionEncoder(embed_dim=EMBED_DIM)

## 9. No-site-embedding ablation

Implements the A5 ablation by replacing the site identity input with zeros, making the model site-agnostic.

In [ ]:
# -----------------------------------------------------------------------
# ABLATION VARIANT: NO SITE EMBEDDING
# Replace site embedding with zeros -- model is site-agnostic
# -----------------------------------------------------------------------
class GenerativeAURORA_NoSite(GenerativeAURORA):
    def __init__(self, T=T_DIFFUSION):
        super().__init__(T=T)

    def forward(self, x0, cond_x, regime, site):
        site_zeros = torch.zeros_like(site)
        return super().forward(x0, cond_x, regime, site_zeros)

    @torch.no_grad()
    def sample(self, cond_x, regime, site, n_scenarios=50):
        site_zeros = torch.zeros_like(site)
        return super().sample(cond_x, regime, site_zeros, n_scenarios)

## 10. Metric functions

Implements the original CRPS, coverage, median MAE, and prediction-interval width calculations used to compare ablation variants.

In [ ]:
# -----------------------------------------------------------------------
# METRICS
# -----------------------------------------------------------------------
def crps_score(actuals, scenarios):
    N, S, H = scenarios.shape
    t1 = np.abs(scenarios - actuals[:, np.newaxis, :]).mean(axis=1)
    t2 = np.zeros((N, H))
    for n in range(N):
        sc = scenarios[n]
        t2[n] = np.abs(sc[:, np.newaxis, :] - sc[np.newaxis, :, :]).mean(axis=(0, 1))
    return float((t1 - 0.5 * t2).mean())


def coverage_from_scenarios(actuals, scenarios, alpha=ALPHA):
    lower = np.quantile(scenarios, alpha / 2,     axis=1)
    upper = np.quantile(scenarios, 1 - alpha / 2, axis=1)
    return float(((actuals >= lower) & (actuals <= upper)).mean()) * 100


def coverage_from_intervals(actuals, lower, upper):
    return float(((actuals >= lower) & (actuals <= upper)).mean()) * 100


def med_mae(actuals, scenarios):
    return float(np.abs(np.median(scenarios, axis=1) - actuals).mean())


def width_from_scenarios(actuals, scenarios, alpha=ALPHA):
    lower = np.quantile(scenarios, alpha / 2,     axis=1)
    upper = np.quantile(scenarios, 1 - alpha / 2, axis=1)
    return float((upper - lower).mean())

## 11. Scenario generation

Generates stochastic scenarios for a model. The optional overrides reproduce the source-script treatment for the no-regime and other ablations.

In [ ]:
# -----------------------------------------------------------------------
# GENERATE SCENARIOS FOR A MODEL
# -----------------------------------------------------------------------
def generate(model, loader, device, scalers, n_sc, regime_override=None, site_override=None):
    model.eval()
    all_act, all_sc, all_reg, all_sit = [], [], [], []

    with torch.no_grad():
        for X, Y, regime, site in loader:
            X      = X.to(device)
            B      = X.shape[0]

            r = torch.zeros(B, dtype=torch.long, device=device) if regime_override == 0 \
                else regime.to(device)
            s = torch.zeros(B, dtype=torch.long, device=device) if site_override == 0 \
                else site.to(device)

            sc = model.sample(X, r, s, n_scenarios=n_sc)
            sc = sc.cpu().numpy().reshape(B, n_sc, HORIZON)

            all_act.append(Y.numpy())
            all_sc.append(sc)
            all_reg.append(regime.numpy())
            all_sit.append(site.numpy())

    act = inv_scale(np.concatenate(all_act, axis=0), scalers)
    sc  = inv_scale(np.concatenate(all_sc,  axis=0), scalers)
    reg = np.concatenate(all_reg, axis=0)
    sit = np.concatenate(all_sit, axis=0)
    return act, sc, reg, sit

## 12. Ablation metric calculation

Combines the source metrics for overall performance and specifically reports cloudy and overcast coverage, using calibrated intervals only when they are explicitly supplied.

In [ ]:
def ablation_metrics(act, sc, reg, cal_lower=None, cal_upper=None):
    crps     = round(crps_score(act, sc), 4)
    cov_all  = round(coverage_from_scenarios(act, sc), 2)
    mae      = round(med_mae(act, sc), 4)
    wid      = round(width_from_scenarios(act, sc), 2)

    # Overcast coverage
    mask_ov  = (reg == 2)
    if mask_ov.sum() > 0:
        if cal_lower is not None and cal_upper is not None:
            cov_ov = round(coverage_from_intervals(act[mask_ov], cal_lower[mask_ov], cal_upper[mask_ov]), 2)
        else:
            cov_ov = round(coverage_from_scenarios(act[mask_ov], sc[mask_ov]), 2)
    else:
        cov_ov = float("nan")

    # Cloudy coverage
    mask_cl = (reg == 1)
    if mask_cl.sum() > 0:
        if cal_lower is not None and cal_upper is not None:
            cov_cl = round(coverage_from_intervals(act[mask_cl], cal_lower[mask_cl], cal_upper[mask_cl]), 2)
        else:
            cov_cl = round(coverage_from_scenarios(act[mask_cl], sc[mask_cl]), 2)
    else:
        cov_cl = float("nan")

    return {
        "CRPS":               crps,
        "Coverage 90% (All)": cov_all if cal_lower is None else round(
            coverage_from_intervals(act, cal_lower, cal_upper), 2),
        "Coverage (Cloudy)":  cov_cl,
        "Coverage (Overcast)": cov_ov,
        "Width 90% PI":       wid if cal_lower is None else round(
            float((cal_upper - cal_lower).mean()), 2),
        "Median MAE":         mae,
    }

## 13. Main setup and data loading

The original `main()` setup loads CPU/CUDA, scalers, train/test datasets, the trained model path, and the calibrated intervals generated by Step 7.

In [ ]:
# -----------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------
def main():
    print("GenerativeAURORA - Step 10: Ablation Study")
    print("")

    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    scalers = load_scalers()
    np.random.seed(42)
    print("Device: " + str(device))
    print("")

    test_ds = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_test.npz"))
    train_ds = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_train.npz"))
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model_path = os.path.join(DATASET_DIR, "best_model.pt")

    # Load calibrated intervals from Step 7
    cal_data  = np.load(os.path.join(EVAL_DIR, "calibrated_intervals.npz"))
    l_cal     = cal_data["lower_cal"]
    u_cal     = cal_data["upper_cal"]
    reg_cal   = cal_data["regimes"]

    results = {}

## 14. A1 — Full model + RCCC

Runs the proposed A1 configuration using the trained full model and RCCC-calibrated intervals.

In [ ]:
    # -----------------------------------------------------------------------
    # A1: Full model + RCCC (proposed)
    # -----------------------------------------------------------------------
    print("A1: GenerativeAURORA + RCCC (proposed)...")

    test_ds = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_test.npz"))
    train_ds = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_train.npz"))
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model_path = os.path.join(DATASET_DIR, "best_model.pt")
    scalers = load_scalers()

    # Load calibrated intervals from Step 7
    cal_data  = np.load(os.path.join(EVAL_DIR, "calibrated_intervals.npz"))
    l_cal     = cal_data["lower_cal"]
    u_cal     = cal_data["upper_cal"]
    reg_cal   = cal_data["regimes"]

    results = {}

    model = GenerativeAURORA(T=T_DIFFUSION).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    act, sc, reg, sit = generate(model, test_loader, device, scalers, N_SCENARIOS)
    results["A1: Full + RCCC (proposed)"] = ablation_metrics(act, sc, reg, l_cal, u_cal)
    print("  CRPS=" + str(results["A1: Full + RCCC (proposed)"]["CRPS"])
          + "  Overcast cov=" + str(results["A1: Full + RCCC (proposed)"]["Coverage (Overcast)"]) + "%")

A1: GenerativeAURORA + RCCC (proposed)...
  CRPS=17.4492  Overcast cov=68.75%


## 15. A2 — Full model, no RCCC

Evaluates the same generated scenarios without RCCC calibration, providing the A2 comparison against A1.

In [ ]:
    # -----------------------------------------------------------------------
    # A2: Full model, no RCCC
    # -----------------------------------------------------------------------
    print("A2: GenerativeAURORA (no RCCC)...")
    results["A2: Full, no RCCC"] = ablation_metrics(act, sc, reg)
    print("  CRPS=" + str(results["A2: Full, no RCCC"]["CRPS"])
          + "  Overcast cov=" + str(results["A2: Full, no RCCC"]["Coverage (Overcast)"]) + "%")

A2: GenerativeAURORA (no RCCC)...
  CRPS=17.4492  Overcast cov=53.12%


## 16. A3 — No regime conditioning

Forces the regime input to Clear for every sample, reproducing the A3 no-regime-conditioning ablation.

In [ ]:
    # -----------------------------------------------------------------------
    # A3: No regime conditioning (force all regime=0)
    # -----------------------------------------------------------------------
    print("A3: DDPM no regime conditioning...")
    act_nr, sc_nr, reg_nr, _ = generate(model, test_loader, device, scalers, N_SCENARIOS,
                                         regime_override=0)
    results["A3: No regime conditioning"] = ablation_metrics(act_nr, sc_nr, reg_nr)
    print("  CRPS=" + str(results["A3: No regime conditioning"]["CRPS"])
          + "  Overcast cov=" + str(results["A3: No regime conditioning"]["Coverage (Overcast)"]) + "%")

A3: DDPM no regime conditioning...
  CRPS=17.4292  Overcast cov=78.12%


## 17. A4 — No condition encoder

Constructs the blind condition-encoder model, loads shared weights with the original `strict=False` setting, and evaluates A4.

In [ ]:
    # -----------------------------------------------------------------------
    # A4: No condition encoder (blind -- no weather context)
    # -----------------------------------------------------------------------
    print("A4: No condition encoder (blind DDPM)...")
    model_blind = GenerativeAURORA_NoContext(T=T_DIFFUSION).to(device)
    model_blind.load_state_dict(
        torch.load(model_path, map_location=device, weights_only=True),
        strict=False   # blind encoder has different architecture, load shared weights only
    )
    act_b, sc_b, reg_b, _ = generate(model_blind, test_loader, device, scalers, N_SCENARIOS)
    results["A4: No condition encoder"] = ablation_metrics(act_b, sc_b, reg_b)
    print("  CRPS=" + str(results["A4: No condition encoder"]["CRPS"])
          + "  Overcast cov=" + str(results["A4: No condition encoder"]["Coverage (Overcast)"]) + "%")

A4: No condition encoder (blind DDPM)...
  CRPS=140.2177  Overcast cov=3.12%


## 18. A5 — No site embedding

Constructs the no-site-embedding model and evaluates A5 using the original implementation.

In [ ]:
    # -----------------------------------------------------------------------
    # A5: No site embedding
    # -----------------------------------------------------------------------
    print("A5: No site embedding...")
    model_ns = GenerativeAURORA_NoSite(T=T_DIFFUSION).to(device)
    model_ns.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    act_ns, sc_ns, reg_ns, _ = generate(model_ns, test_loader, device, scalers, N_SCENARIOS)
    results["A5: No site embedding"] = ablation_metrics(act_ns, sc_ns, reg_ns)
    print("  CRPS=" + str(results["A5: No site embedding"]["CRPS"])
          + "  Overcast cov=" + str(results["A5: No site embedding"]["Coverage (Overcast)"]) + "%")

A5: No site embedding...
  CRPS=21.0947  Overcast cov=50.0%


## 19. A6 — Gaussian baseline

Generates the Gaussian lower-bound baseline from training-set mean and standard deviation, exactly as implemented in the source script.

In [ ]:
    # -----------------------------------------------------------------------
    # A6: Gaussian baseline (lower bound)
    # -----------------------------------------------------------------------
    print("A6: Gaussian baseline...")
    train_Y_wm2 = inv_scale(train_ds.Y.numpy(), scalers)
    test_Y_wm2  = inv_scale(test_ds.Y.numpy(),  scalers)
    mu  = train_Y_wm2.mean(axis=0)
    sig = np.maximum(train_Y_wm2.std(axis=0), 1.0)
    N   = len(test_Y_wm2)
    sc_g = np.random.normal(
        mu[np.newaxis, np.newaxis, :],
        sig[np.newaxis, np.newaxis, :],
        size=(N, N_SCENARIOS, HORIZON)
    ).clip(0, None).astype(np.float32)
    results["A6: Gaussian baseline"] = ablation_metrics(test_Y_wm2, sc_g, test_ds.regime.numpy())
    print("  CRPS=" + str(results["A6: Gaussian baseline"]["CRPS"])
          + "  Overcast cov=" + str(results["A6: Gaussian baseline"]["Coverage (Overcast)"]) + "%")

A6: Gaussian baseline...
  CRPS=97.4716  Overcast cov=50.0%


## 20. Build Table VI

Builds and saves the ablation results as Table VI in both CSV and LaTeX formats, using the original table formatting logic.

In [ ]:
    # -----------------------------------------------------------------------
    # BUILD ABLATION TABLE
    # -----------------------------------------------------------------------
    print("")
    print("=== Ablation Table (Table VI) ===")
    table_df = pd.DataFrame(results).T
    print(table_df.to_string())

    # Save CSV
    csv_path = os.path.join(TABLE_DIR, "table6_ablation.csv")
    table_df.to_csv(csv_path, encoding="utf-8")
    print("")
    print("Saved: " + csv_path)

    # Save LaTeX
    lines = [
        "\\begin{table}[ht]",
        "\\centering",
        "\\caption{Ablation study. Each row removes one component from the full GenerativeAURORA + RCCC model. "
        "Best result per column in bold. Overcast coverage is the key diagnostic metric.}",
        "\\label{tab:ablation}",
        "\\begin{tabular}{lrrrrrr}",
        "\\toprule",
        "Variant & CRPS & Cov. 90\\% & Cov. Cloudy & Cov. Overcast & Width & MAE \\\\",
        "\\midrule",
    ]
    for name, row in results.items():
        short_name = name.split(":")[1].strip()
        is_proposed = "proposed" in name
        prefix = "\\textbf{" if is_proposed else ""
        suffix = "}" if is_proposed else ""
        line = (prefix + short_name + suffix + " & "
                + prefix + str(row["CRPS"]) + suffix + " & "
                + prefix + str(row["Coverage 90% (All)"]) + "\\%" + suffix + " & "
                + prefix + str(row["Coverage (Cloudy)"]) + "\\%" + suffix + " & "
                + prefix + str(row["Coverage (Overcast)"]) + "\\%" + suffix + " & "
                + prefix + str(row["Width 90% PI"]) + suffix + " & "
                + prefix + str(row["Median MAE"]) + suffix + " \\\\")
        lines.append(line)

    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    tex_content = "\n".join(lines)

    tex_path = os.path.join(TABLE_DIR, "table6_ablation.tex")
    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(tex_content)
    print("Saved: " + tex_path)


=== Ablation Table (Table VI) ===
                                CRPS  Coverage 90% (All)  Coverage (Cloudy)  Coverage (Overcast)  Width 90% PI  Median MAE
A1: Full + RCCC (proposed)   17.4492               95.32              93.89                68.75        120.72     23.5448
A2: Full, no RCCC            17.4492               92.23              86.96                53.12        101.80     23.5448
A3: No regime conditioning   17.4292               92.17              89.95                78.12        102.19     23.5021
A4: No condition encoder    140.2177               16.75              17.80                 3.12        115.12    156.9758
A5: No site embedding        21.0947               84.47              80.98                50.00         93.92     28.2296
A6: Gaussian baseline        97.4716               91.11              61.55                50.00        507.03    143.7806

Saved: /content/drive/MyDrive/Colab Notebooks/Dataset/paper_tables/table6_ablation.csv
Saved: /content/

## 21. Paper-ready ablation narrative

Prints the paper-ready narrative comparisons requested by the original script, including RCCC gain and the effects of removing regime conditioning and the condition encoder.

In [ ]:
    # -----------------------------------------------------------------------
    # PRINT PAPER-READY SUMMARY
    # -----------------------------------------------------------------------
    print("")
    print("=== Paper narrative (Section V, ablation paragraph) ===")
    print("")
    a1 = results["A1: Full + RCCC (proposed)"]
    a2 = results["A2: Full, no RCCC"]
    a3 = results["A3: No regime conditioning"]
    a4 = results["A4: No condition encoder"]

    print("Removing RCCC (A2 vs A1):")
    print("  Overall coverage: " + str(a2["Coverage 90% (All)"]) + "% vs "
          + str(a1["Coverage 90% (All)"]) + "%")
    print("  Overcast coverage: " + str(a2["Coverage (Overcast)"]) + "% vs "
          + str(a1["Coverage (Overcast)"]) + "%")
    delta_ov = round(a1["Coverage (Overcast)"] - a2["Coverage (Overcast)"], 2)
    print("  Overcast gain from RCCC: +" + str(delta_ov) + " pp")

    print("")
    print("Removing regime conditioning (A3 vs A2):")
    delta_crps = round(a2["CRPS"] - a3["CRPS"], 4)
    sign = "+" if delta_crps >= 0 else ""
    print("  CRPS change: " + sign + str(delta_crps)
          + "  (small because 93% data is Clear sky)")

    print("")
    print("Removing condition encoder (A4 vs A2):")
    delta_crps_blind = round(a2["CRPS"] - a4["CRPS"], 4)
    sign2 = "+" if delta_crps_blind >= 0 else ""
    print("  CRPS change: " + sign2 + str(delta_crps_blind)
          + "  (weather context contribution)")

    print("")
    print("=== Step 10 complete ===")
    print("Next: run step11_final_figures.py")


=== Paper narrative (Section V, ablation paragraph) ===

Removing RCCC (A2 vs A1):
  Overall coverage: 92.23% vs 95.32%
  Overcast coverage: 53.12% vs 68.75%
  Overcast gain from RCCC: +15.63 pp

Removing regime conditioning (A3 vs A2):
  CRPS change: +0.02  (small because 93% data is Clear sky)

Removing condition encoder (A4 vs A2):
  CRPS change: -122.7685  (weather context contribution)

=== Step 10 complete ===
Next: run step11_final_figures.py


## 22. Script entry point

Preserves the final `__main__` execution block from the source script.

In [ ]:
if __name__ == "__main__":
    main()

GenerativeAURORA - Step 10: Ablation Study

Device: cuda



## Execution note

The original Windows paths are intentionally preserved:

`C:/Users/user/Paper_10/Dataset`

`C:/Users/user/Paper_10/Coding`

For actual Colab execution, these paths must point to the corresponding dataset and source-code locations.

### Main inputs
- `sequences_train.npz`
- `sequences_test.npz`
- `scaler_params.json`
- `best_model.pt`
- `evaluation/calibrated_intervals.npz`
- `step4_build_model.py`

### Main outputs
- `paper_tables/table6_ablation.csv`
- `paper_tables/table6_ablation.tex`

The notebook also prints the paper-ready ablation narrative from the original script.
